In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import importlib

import utils

results_dir = Path() / "results" / "ray_sampling"
figures_dir = Path() / "figures"

method_to_label = {
    "iarap": "Ours",
    "ling": "Ling et al. (numpy reference)",
    "ling_pytorch": "Ling et al. (PyTorch port)",
}

In [ ]:
cache = True # Set to True to skip already benchmarked configurations
methods = ["iarap", "ling_pytorch", "ling"]
num_rays = [100, 1000, 10000, 100000, 1000000, 10000000]

results_dir.mkdir(parents=True, exist_ok=True)

for method in methods:
    for num_ray in num_rays:
        output_path = results_dir / f"{method}_{num_ray}.csv"
        if output_path.exists() and cache:
            print(f"Skipping {method} with {num_ray} rays (cached).")
            continue
        
        utils.call_script(Path().absolute() / "benchmark_ray_sampling.py", method=method, num_rays=num_ray, num_runs=5, num_warmup_runs=5, dim=3, output=results_dir / f"{method}_{num_ray}.csv")

In [ ]:
importlib.reload(utils)

# Load and merge all dataframes
dataframes = [pd.read_csv(f) for f in results_dir.glob("*.csv")]
df_raw = pd.concat(dataframes, ignore_index=True)

# Compute averages and standard deviations over runs, and merge them into a single dataframe
df_avg = df_raw.groupby(["method", "num_rays"]).agg({"elapsed_time": "mean", "peak_memory": "mean"}).reset_index()
df_std = df_raw.groupby(["method", "num_rays"]).agg({"elapsed_time": "std", "peak_memory": "std"}).reset_index()
df = pd.merge(df_avg, df_std, on=["method", "num_rays"], suffixes=("_avg", "_std"))

# Create a gridspec so the two plots share the same x-axis cleanly
fig = plt.figure(figsize=(10, 5))
gs = fig.add_gridspec(2, 1, hspace=0.08)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax1.tick_params(labelbottom=False)

utils.set_figure_title_with_specs(fig, benchmark_name="Ray Sampling")
utils.plot_bar_chart(ax1, df, metric="elapsed_time", metric_label="Runtime (ms)", method_to_label=method_to_label, reference_method="ling_pytorch", y_quantile=0.7, x_label=False)
utils.plot_bar_chart(ax2, df, metric="peak_memory", metric_label="Peak memory (MiB)", method_to_label=method_to_label, reference_method="ling_pytorch", y_quantile=0.8, legend=False, ignored_methods=["ling"])

figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / "benchmark_ray_sampling.png", dpi=150, bbox_inches="tight")